# AutoSort Evaluation Pipeline

Main steps:
1. Load data (same as training)
2. Prepare evaluation data for each time segment
3. Load models and evaluate for each clique and time segment


In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')
import os
import gc
import torch
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
from spikeinterface.core import concatenate_recordings
from probeinterface import write_probeinterface, read_probeinterface
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from utils_clique import (
    prepare_training_data,
    evaluate_autosort_model,
    build_sliding_cliques,
    CliqueInfo,
    visualize_umap_features,
    calibration_model,
    real_time_processing,
    generate_confusion_matrix_df,
    compute_noise_detection_metrics,
    match_neurons,
    SimpleAutoSort,
    SimpleWaveformLoader
)


In [24]:
# Load data (same as training)
recording_path = "/media/ubuntu/sda/mouse_test/raw_data/WLF_128chmouse1_ASDstim_251203_194748"
spike_inf_path = "/media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_ASDstim_251203_194748_10k_kilosort/spike_inf.tsv"
neuron_inf_path = "/media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_ASDstim_251203_194748_10k_kilosort/neuron_inf.pkl"

# Load recording (same as training - Intan format)
file_list = os.listdir(recording_path)
file_list.remove("settings.xml")
file_list.remove("log_200846.csv")

file_list = sorted(file_list)
recording_raw_list = []
for file in file_list:
    recording_raw_list.append(se.read_intan(f"{recording_path}/{file}", stream_id= '0'))
recording = concatenate_recordings(recording_list=recording_raw_list)
recording_raw = spre.unsigned_to_signed(recording)
recording_raw = spre.resample(recording_raw, 10000)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

# Note: Whitening will be applied in calibration_model based on first 60s of data
# The whitening matrix will be computed from calibration data and reused for real-time processing

# Load probe (same as training)
probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
if probe is None:
    raise ValueError("Recording does not have probe information")

# 定义按shank构建cliques的函数（same as training）
def build_shank_cliques(probe, shank_boundaries=[250, 750, 1250]):
    """
    根据x坐标划分shank并构建cliques
    
    Parameters:
        probe: Probe对象
        shank_boundaries: shank之间的x坐标边界，默认[250, 750, 1250]
                         将probe划分为4个shank:
                         - shank 0: x < 250
                         - shank 1: 250 <= x < 750
                         - shank 2: 750 <= x < 1250
                         - shank 3: x >= 1250
    
    Returns:
        cliques: List[CliqueInfo] - 每个shank对应一个clique
    """
    from typing import List
    
    df = probe.to_dataframe()
    if "device_channel_indices" in df.columns:
        device_indices = df["device_channel_indices"].astype(int).to_numpy()
    else:
        device_indices = np.arange(len(df), dtype=int)
    positions = df.loc[:, ["x", "y"]].to_numpy()
    contact_ids = df["contact_ids"].astype(str).to_numpy()
    
    # 根据x坐标划分shank
    x_coords = positions[:, 0]
    shank_boundaries_sorted = sorted(shank_boundaries)
    
    cliques: List[CliqueInfo] = []
    
    # 定义shank范围
    shank_ranges = [
        (float('-inf'), shank_boundaries_sorted[0]),  # shank 0: x < 250
        (shank_boundaries_sorted[0], shank_boundaries_sorted[1]),  # shank 1: 250 <= x < 750
        (shank_boundaries_sorted[1], shank_boundaries_sorted[2]),  # shank 2: 750 <= x < 1250
        (shank_boundaries_sorted[2], float('inf')),  # shank 3: x >= 1250
    ]
    
    for shank_id, (x_min, x_max) in enumerate(shank_ranges):
        # 找到属于当前shank的通道
        if x_min == float('-inf'):
            mask = x_coords < x_max
        elif x_max == float('inf'):
            mask = x_coords >= x_min
        else:
            mask = (x_coords >= x_min) & (x_coords < x_max)
        
        shank_device_indices = device_indices[mask]
        shank_contact_ids = contact_ids[mask]
        shank_positions = positions[mask]
        
        if len(shank_device_indices) == 0:
            print(f"[WARNING] Shank {shank_id} has no channels")
            continue
        
        # 计算shank的中心位置
        center = tuple(np.mean(shank_positions, axis=0))
        
        # 创建CliqueInfo对象
        clique = CliqueInfo(
            clique_id=shank_id,
            device_channel_indices=list(shank_device_indices),
            contact_ids=list(shank_contact_ids),
            center=center,
        )
        cliques.append(clique)
        
        print(f"[INFO] Shank {shank_id}: {len(shank_device_indices)} channels "
              f"(x range: {x_min if x_min != float('-inf') else 'min'} to "
              f"{x_max if x_max != float('inf') else 'max'})")
    
    print(f"[INFO] Built {len(cliques)} cliques from {len(shank_boundaries) + 1} shanks")
    return cliques

# Build cliques from probe by shank (same as training)
cliques = build_shank_cliques(probe, shank_boundaries=[250, 750, 1250])

# Load clique information (saved during training)
base_save_dir = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128/output"
clique_info_path = Path(base_save_dir) / "clique_info.pkl"

if clique_info_path.exists():
    with open(clique_info_path, 'rb') as f:
        clique_info = pickle.load(f)
    # Use cliques from saved file if available, otherwise use the ones we just built
    if 'cliques' in clique_info:
        cliques = clique_info['cliques']
        print(f"Loaded {len(cliques)} cliques from {clique_info_path}")
    else:
        print(f"Using built cliques: {len(cliques)} cliques")
else:
    print(f"Clique info not found at {clique_info_path}, using built cliques: {len(cliques)} cliques")

# Load GT data (evaluation data)
spike_inf = None
neuron_inf = None

if Path(spike_inf_path).exists():
    spike_inf = pd.read_csv(spike_inf_path, sep='\t', index_col=0)
else:
    raise ValueError(f"spike_inf not found at {spike_inf_path}")

if Path(neuron_inf_path).exists():
    with open(neuron_inf_path, 'rb') as f:
        neuron_inf = pickle.load(f)
else:
    raise ValueError(f"neuron_inf.pkl not found at {neuron_inf_path}")

# Load training data neuron_inf (for calibration matching)
# Training data path (same recording as training, but different session)
train_neuron_inf_path = "/media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_natima_RHD_251129_183351_10k_kilosort/neuron_inf.pkl"
train_neuron_inf = None

if Path(train_neuron_inf_path).exists():
    with open(train_neuron_inf_path, 'rb') as f:
        train_neuron_inf = pickle.load(f)
    print(f"Training neuron_inf loaded from: {train_neuron_inf_path}")
    print(f"  Number of training neurons: {len(train_neuron_inf)}")
else:
    print(f"Warning: Training neuron_inf not found at {train_neuron_inf_path}")
    print(f"  Will use evaluation neuron_inf as fallback (may not have position/waveform info)")

print(f"\nRecording loaded successfully")
print(f"Sampling rate: {recording_f.get_sampling_frequency()} Hz")
print(f"Number of channels: {recording_f.get_num_channels()}")
print(f"Recording duration: {recording_f.get_num_samples() / recording_f.get_sampling_frequency():.2f} seconds")


[INFO] Shank 0: 32 channels (x range: min to 250)
[INFO] Shank 1: 32 channels (x range: 250 to 750)
[INFO] Shank 2: 32 channels (x range: 750 to 1250)
[INFO] Shank 3: 32 channels (x range: 1250 to max)
[INFO] Built 4 cliques from 4 shanks
Loaded 4 cliques from /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128/output/clique_info.pkl
Training neuron_inf loaded from: /media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_natima_RHD_251129_183351_10k_kilosort/neuron_inf.pkl
  Number of training neurons: 33

Recording loaded successfully
Sampling rate: 10000.0 Hz
Number of channels: 128
Recording duration: 1313.36 seconds


In [25]:
# Step 2.5: Match neurons between training and evaluation data
# This establishes the mapping between training set neurons and evaluation set neurons
# based on position and waveform similarity

print(f"\n{'='*80}")
print(f"Step 2.5: Neuron Matching (Training Set <-> Evaluation Set)")
print(f"{'='*80}")

# Check if neuron_inf has required columns for matching
required_cols = ['position_1', 'position_2', 'position_waveform']
train_has_cols = all(col in train_neuron_inf.columns for col in required_cols) if train_neuron_inf is not None else False
eval_has_cols = all(col in neuron_inf.columns for col in required_cols)

if train_has_cols and eval_has_cols and train_neuron_inf is not None:
    # Perform neuron matching
    eval_neuron_inf_matched = match_neurons(
        train_neuron_inf=train_neuron_inf,
        eval_neuron_inf=neuron_inf,
        position_threshold=10,
        waveform_similarity_threshold=0.95
    )
    
    # Count matched and unmatched neurons
    matched_neurons = eval_neuron_inf_matched[eval_neuron_inf_matched['neuron_match'] != 'unmatch']
    unmatched_neurons = eval_neuron_inf_matched[eval_neuron_inf_matched['neuron_match'] == 'unmatch']
    
    print(f"\nMatching Summary:")
    print(f"  - Total evaluation neurons: {len(eval_neuron_inf_matched)}")
    print(f"  - Matched neurons: {len(matched_neurons)}")
    print(f"  - Unmatched neurons: {len(unmatched_neurons)}")
    
    if len(matched_neurons) > 0:
        print(f"  - Matched neuron pairs (first 5):")
        for idx, row in matched_neurons.head(5).iterrows():
            print(f"    {row['Neuron']} -> {row['neuron_match']}")
    
    if len(unmatched_neurons) > 0:
        print(f"  - Unmatched neuron names: {unmatched_neurons['Neuron'].tolist()}")
    
    # Use matched neuron_inf for subsequent processing
    neuron_inf = eval_neuron_inf_matched
    print(f"\n  ✓ Using matched neuron_inf for calibration and evaluation")
    print(f"  ✓ GT label mapping will be established using neuron_match column")
else:
    print(f"Warning: Missing required columns or training neuron_inf for neuron matching")
    if train_neuron_inf is None:
        print(f"  - Training neuron_inf is None")
    else:
        print(f"  - Training neuron_inf has required columns: {train_has_cols}")
    print(f"  - Evaluation neuron_inf has required columns: {eval_has_cols}")
    print(f"  - Required columns: {required_cols}")
    print(f"  - Will proceed without matching (neuron_match column will not be available)")
    print(f"  - Note: GT label mapping may not be established correctly")



Step 2.5: Neuron Matching (Training Set <-> Evaluation Set)
Neuron Matching
Matching neurons...

Matching completed:
  - Total evaluation neurons: 30
  - Matched: 0
  - Unmatched: 30

Matching Summary:
  - Total evaluation neurons: 30
  - Matched neurons: 0
  - Unmatched neurons: 30
  - Unmatched neuron names: ['Neuron_24', 'Neuron_38', 'Neuron_41', 'Neuron_48', 'Neuron_52', 'Neuron_88', 'Neuron_90', 'Neuron_91', 'Neuron_106', 'Neuron_113', 'Neuron_116', 'Neuron_118', 'Neuron_121', 'Neuron_123', 'Neuron_125', 'Neuron_130', 'Neuron_133', 'Neuron_137', 'Neuron_139', 'Neuron_140', 'Neuron_148', 'Neuron_150', 'Neuron_154', 'Neuron_156', 'Neuron_158', 'Neuron_160', 'Neuron_162', 'Neuron_164', 'Neuron_166', 'Neuron_168']

  ✓ Using matched neuron_inf for calibration and evaluation
  ✓ GT label mapping will be established using neuron_match column


In [11]:
# Parameters for calibration and real-time processing
calibration_duration_seconds = 200  # Calibration stage duration (first 60 seconds)
n_additional_clusters = 5  # Number of additional clusters for K-means
time_window_seconds = 10.0  # Real-time processing window size (seconds)
total_duration_seconds = None  # Total processing duration (None = process until end)

# MountainSort4 detection parameters (same as training)
mountainsort4_detection_params = {
    'detect_threshold': 3.5,     
    'detect_interval': 60,          # 最小检测间隔（samples）
    'detect_sign': -1,             # -1: 检测负峰值, 0: 双向, 1: 正峰值
    'margin': 0,                   # 边界margin（samples）
}

# Window parameters (same as training)
window_params = {
    'left_sample': 10,
    'right_sample': 20,
}

# Calibration matching parameters
position_threshold = 10.0  # Position distance threshold (microns)
waveform_similarity_threshold = 0.95  # Waveform similarity threshold

# Number of training runs per clique
n_runs = 1

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Build device to recording channel mapping (same as training)
recording_channel_ids = recording_f.get_channel_ids()
probe_df = probe.to_dataframe()
if "device_channel_indices" in probe_df.columns:
    probe_device_indices = probe_df["device_channel_indices"].astype(int).to_numpy()
else:
    probe_device_indices = np.arange(len(probe_df), dtype=int)

device_to_recording_channel = {}
for i, device_idx in enumerate(probe_device_indices):
    if i < len(recording_channel_ids):
        device_to_recording_channel[device_idx] = recording_channel_ids[i]

print(f"\n{'='*80}")
print(f"Parameters:")
print(f"{'='*80}")
print(f"Calibration duration: {calibration_duration_seconds} seconds")
print(f"Number of additional clusters: {n_additional_clusters}")
print(f"Real-time processing window: {time_window_seconds} seconds")
print(f"Total processing duration: {total_duration_seconds if total_duration_seconds is not None else 'until end'}")
print(f"Number of runs per clique: {n_runs}")
print(f"{'='*80}")


Using device: cuda

Parameters:
Calibration duration: 200 seconds
Number of additional clusters: 5
Real-time processing window: 10.0 seconds
Total processing duration: until end
Number of runs per clique: 1


In [12]:
# Store all calibration and real-time processing results
all_calibration_results = {}  # {clique_id: {run_id: calibration_results}}
all_processing_results = {}  # {clique_id: {run_id: processing_results}}

# Process each clique
for clique_id in range(len(cliques)):
    print(f"\n{'='*80}")
    print(f"Processing Clique {clique_id:02d}")
    print(f"{'='*80}")
    if clique_id in [0, 2]:
        continue
    
    clique = cliques[clique_id]
    n_channels = len(clique.device_channel_indices)
    
    # Map device indices to recording channel IDs (same as training)
    clique_device_indices = set(clique.device_channel_indices)
    sorted_device_indices = sorted(clique_device_indices)
    clique_channel_ids = [device_to_recording_channel[idx] for idx in sorted_device_indices if idx in device_to_recording_channel]
    
    # Create recording_clique (subset of channels for this clique)
    recording_clique = recording_f.select_channels(channel_ids=clique_channel_ids)
    recording_clique = recording_clique.rename_channels(sorted_device_indices)
    
    # Debug: Check recording_clique duration
    sampling_rate = recording_clique.get_sampling_frequency()
    total_samples = recording_clique.get_num_samples()
    total_seconds = total_samples / sampling_rate
    print(f"  Recording_clique duration: {total_samples} samples ({total_seconds:.2f} seconds)")
    print(f"  Expected for calibration: {int(calibration_duration_seconds * sampling_rate)} samples ({calibration_duration_seconds} seconds)")
    
    # Filter GT data for this clique
    # Get neurons that have channels fully inside this clique (best_channels -> channel_id -> tract_channel fallback)
    neuron_inf_clique_list = []
    import ast
    for _, row in neuron_inf.iterrows():
        channels = row.get('best_channels', [])
        if isinstance(channels, str):
            try:
                channels = ast.literal_eval(channels)
            except Exception:
                channels = []
        if not isinstance(channels, (list, tuple, np.ndarray)) or len(channels) == 0:
            channels = row.get('channel_id', [])
            if isinstance(channels, str):
                try:
                    channels = ast.literal_eval(channels)
                except Exception:
                    channels = []
            if not isinstance(channels, (list, tuple, np.ndarray)) or len(channels) == 0:
                if 'tract_channel' in row:
                    tract_ch = row.get('tract_channel', None)
                    if pd.notna(tract_ch) and tract_ch is not None:
                        channels = [int(tract_ch)]
                else:
                    channels = []
        channels_set = set(channels)
        if len(channels_set) == 0:
            continue
        if channels_set.issubset(clique_device_indices):
            neuron_inf_clique_list.append(row)
    neuron_inf_clique = pd.DataFrame(neuron_inf_clique_list).reset_index(drop=True)
    
    # Filter spike_inf accordingly (only spikes whose neuron/cluster in neuron_inf_clique)
    spike_inf_clique = spike_inf.copy()
    clique_neuron_names = set(neuron_inf_clique['Neuron'].unique()) if len(neuron_inf_clique) > 0 else set()
    if 'neuron' in spike_inf_clique.columns:
        spike_inf_clique = spike_inf_clique[spike_inf_clique['neuron'].isin(clique_neuron_names)].copy()
    elif 'cluster' in spike_inf_clique.columns:
        spike_inf_clique = spike_inf_clique[spike_inf_clique['cluster'].isin(clique_neuron_names)].copy()
    else:
        raise ValueError("spike_inf must have either 'neuron' or 'cluster' column")
    
    all_calibration_results[clique_id] = {}
    all_processing_results[clique_id] = {}
    
    # Process each training run
    for run_id in range(1, n_runs + 1):
        print(f"\n{'-'*60}")
        print(f"Clique {clique_id:02d} - Run {run_id}/{n_runs}")
        print(f"{'-'*60}")
        
        # Model save directory for this clique and run
        model_save_dir = Path(base_save_dir) / f"clique_{clique_id:02d}" / "model_save" / f"run_{run_id}"
        
        if not model_save_dir.exists():
            print(f"  Model not found at {model_save_dir}, skipping...")
            continue
        
        try:
            # Load model
            print(f"  Loading model...")
            train_data_dir = Path(base_save_dir) / f"clique_{clique_id:02d}" / "train_data"
            
            # Load keep_id
            keep_id_path = model_save_dir / 'keep_id.pkl'
            if not keep_id_path.exists():
                print(f"  Warning: keep_id.pkl not found, skipping...")
                continue
            
            with open(keep_id_path, 'rb') as f:
                keep_id = pickle.load(f)
            
            # Create dataset to get weights (for model initialization)
            dataset = SimpleWaveformLoader(
                root=str(train_data_dir) + '/',
                shank_channel=np.arange(n_channels),
                Keep_id=keep_id
            )
            
            # Create model
            autosort_model = SimpleAutoSort(
                ch_num=n_channels,
                samplepoints=window_params['left_sample'] + window_params['right_sample'],
                device=device,
                set_shank_id=keep_id,
                save_dir=str(model_save_dir) + "/",
                pos_weight_noise=dataset.pos_weight_noise.to(device),
                pos_weight_label=dataset.pos_weight_label.to(device)
            )
            
            # Load model weights
            autosort_model.load_model()
            autosort_model.eval()
            
            # Load training neuron_inf (for calibration matching)
            # Get neurons from training data neuron_mapping (these are the neurons for this clique)
            train_data_dir = Path(base_save_dir) / f"clique_{clique_id:02d}" / "train_data"
            neuron_mapping_path = train_data_dir / "neuron_mapping.pkl"
            
            if neuron_mapping_path.exists():
                with open(neuron_mapping_path, 'rb') as f:
                    train_neuron_mapping = pickle.load(f)
                train_neuron_list = train_neuron_mapping.get('unique_neurons', [])
                print(f"  Training data has {len(train_neuron_list)} neurons for this clique")
                
                # Filter train_neuron_inf to only include neurons in this clique
                if train_neuron_inf is not None:
                    # Filter train_neuron_inf to only include neurons in train_neuron_list
                    train_neuron_inf_clique = train_neuron_inf[
                        train_neuron_inf['Neuron'].isin(train_neuron_list)
                    ].copy()
                    # 进一步过滤：仅保留 best_channels（或 fallback）全部落在本 clique 的 neuron
                    filtered_rows = []
                    import ast
                    for _, r in train_neuron_inf_clique.iterrows():
                        chs = r.get('best_channels', [])
                        if isinstance(chs, str):
                            try:
                                chs = ast.literal_eval(chs)
                            except Exception:
                                chs = []
                        if not isinstance(chs, (list, tuple, np.ndarray)) or len(chs) == 0:
                            chs = r.get('channel_id', [])
                            if isinstance(chs, str):
                                try:
                                    chs = ast.literal_eval(chs)
                                except Exception:
                                    chs = []
                            if not isinstance(chs, (list, tuple, np.ndarray)) or len(chs) == 0:
                                if 'tract_channel' in r:
                                    trc = r.get('tract_channel', None)
                                    if pd.notna(trc) and trc is not None:
                                        chs = [int(trc)]
                                else:
                                    chs = []
                        chs_set = set(chs)
                        if len(chs_set) > 0 and chs_set.issubset(clique_device_indices):
                            filtered_rows.append(r)
                    train_neuron_inf_clique = pd.DataFrame(filtered_rows).reset_index(drop=True)
                    print(f"  Filtered training neuron_inf: {len(train_neuron_inf_clique)} neurons (from {len(train_neuron_inf)} total)")
                    
                    if len(train_neuron_inf_clique) == 0:
                        print(f"  Warning: No matching neurons found in train_neuron_inf, using eval neuron_inf as fallback")
                        train_neuron_inf_clique = neuron_inf_clique.copy()
                else:
                    # Fallback: use eval neuron_inf (may not have position/waveform info)
                    print(f"  Warning: train_neuron_inf not available, using evaluation neuron_inf as fallback")
                    train_neuron_inf_clique = neuron_inf_clique.copy()
            else:
                # Fallback: use eval neuron_inf (may not have position/waveform info)
                print(f"  Warning: neuron_mapping.pkl not found, using evaluation neuron_inf as fallback")
                train_neuron_inf_clique = neuron_inf_clique.copy()
            
            # Calculate valid_channels: union of all best_channels for TRAINING SET neurons in this clique
            # IMPORTANT: valid_channels should be based on training set neurons, not evaluation set neurons
            # This ensures detection is only performed on channels where the model was trained to detect spikes
            
            # Map probe device_channel_indices to clique channel indices
            probe_to_clique_index = {}
            for clique_idx, device_idx in enumerate(sorted_device_indices):
                probe_to_clique_index[device_idx] = clique_idx
            
            # Collect all best_channels from TRAINING SET neurons in this clique（仅当全部通道都在当前clique内）
            all_best_channels_clique_indices = set()
            
            # Use train_neuron_inf_clique (training set neurons) to calculate valid_channels
            if len(train_neuron_inf_clique) > 0:
                for idx, row in train_neuron_inf_clique.iterrows():
                    channels = row.get('best_channels', [])
                    if isinstance(channels, str):
                        import ast
                        try:
                            channels = ast.literal_eval(channels)
                        except:
                            channels = []
                    if not isinstance(channels, (list, tuple, np.ndarray)) or len(channels) == 0:
                        channels = row.get('channel_id', [])
                        if isinstance(channels, str):
                            import ast
                            try:
                                channels = ast.literal_eval(channels)
                            except:
                                channels = []
                        if not isinstance(channels, (list, tuple, np.ndarray)) or len(channels) == 0:
                            if 'tract_channel' in row:
                                tract_ch = row.get('tract_channel', None)
                                if pd.notna(tract_ch) and tract_ch is not None:
                                    channels = [int(tract_ch)]
                            else:
                                channels = []
                    channels_set = set(channels)
                    if len(channels_set) == 0 or (not channels_set.issubset(clique_device_indices)):
                        continue
                    # Map probe channel indices to clique channel indices；要求全部能映射
                    mapped = []
                    for probe_ch_idx in channels_set:
                        if probe_ch_idx in probe_to_clique_index:
                            mapped.append(probe_to_clique_index[probe_ch_idx])
                    if len(mapped) != len(channels_set):
                        continue
                    all_best_channels_clique_indices.update(mapped)
                
                valid_channels = sorted(list(all_best_channels_clique_indices)) if len(all_best_channels_clique_indices) > 0 else None
                
            else:
                # Fallback: if no training set neurons available, use all channels (not ideal)
                print(f"  Warning: train_neuron_inf_clique is empty, cannot determine valid_channels from training set")
                print(f"  Will detect on all channels (this may not be optimal)")
                valid_channels = None
            
            # Set up evaluation results save directory
            eval_save_dir = model_save_dir / "eval"
            eval_save_dir.mkdir(parents=True, exist_ok=True)
            run_name = f"run_{run_id}"
            
            # Generate date string (optional, can be customized)
            from datetime import datetime
            date_str = datetime.now().strftime("%m%d%y")  # Format: MMDDYY
            
            calibration_results = calibration_model(
                recording_f=recording_clique,
                autosort_model=autosort_model,
                train_neuron_inf=train_neuron_inf_clique,
                probe=probe,  # Pass probe object directly (not from recording_f)
                calibration_duration_seconds=calibration_duration_seconds,
                n_additional_clusters=n_additional_clusters,
                detect_threshold=mountainsort4_detection_params['detect_threshold'],
                detect_interval=mountainsort4_detection_params['detect_interval'],
                detect_sign=mountainsort4_detection_params['detect_sign'],
                margin=mountainsort4_detection_params['margin'],
                window_params=window_params,
                position_threshold=position_threshold,
                waveform_similarity_threshold=waveform_similarity_threshold,
                eval_neuron_inf=neuron_inf_clique,
                eval_spike_inf=spike_inf_clique,
                valid_channels=valid_channels,
                device=device,
                save_eval_results=True,  # Enable saving evaluation results
                eval_save_dir=str(eval_save_dir),  # Save directory
                run_name=run_name,  # Run name
                train_data_dir=str(train_data_dir),  # Training data directory to load whitening matrix
                date_str=date_str,  # Date string for file naming
                skip_noise_classifier=False
            )
            
            all_calibration_results[clique_id][run_id] = calibration_results
            
            # Save calibration results
            calibration_save_path = model_save_dir / "calibration_results.pkl"
            with open(calibration_save_path, 'wb') as f:
                pickle.dump(calibration_results, f)
            print(f"  Calibration results saved to: {calibration_save_path}")
            
            gc.collect()
            torch.cuda.empty_cache()
            # # Stage 2: Real-time Processing
            # print(f"\n  {'='*50}")
            # print(f"  Stage 2: Real-time Processing")
            # print(f"  {'='*50}")
            
            # processing_results = real_time_processing(
            #     recording_f=recording_clique,
            #     autosort_model=autosort_model,
            #     calibration_results=calibration_results,
            #     start_time_seconds=calibration_duration_seconds,
            #     time_window_seconds=time_window_seconds,
            #     total_duration_seconds=total_duration_seconds,
            #     detect_threshold=mountainsort4_detection_params['detect_threshold'],
            #     detect_interval=mountainsort4_detection_params['detect_interval'],
            #     detect_sign=mountainsort4_detection_params['detect_sign'],
            #     margin=mountainsort4_detection_params['margin'],
            #     window_params=window_params,
            #     eval_neuron_inf=neuron_inf_clique,
            #     eval_spike_inf=spike_inf_clique,
            #     device=device,
            # )
            
            # all_processing_results[clique_id][run_id] = processing_results
            
            # # Save processing results
            # processing_save_path = model_save_dir / "real_time_processing_results.pkl"
            # with open(processing_save_path, 'wb') as f:
            #     pickle.dump(processing_results, f)
            # print(f"  Real-time processing results saved to: {processing_save_path}")
            
            # print(f"\n  Clique {clique_id:02d} - Run {run_id} completed!")
            
        except Exception as e:
            print(f"  Error processing clique {clique_id}, run {run_id}: {e}")
            import traceback
            traceback.print_exc()
        break



Processing Clique 00

Processing Clique 01
  Recording_clique duration: 81786432 samples (8178.64 seconds)
  Expected for calibration: 2000000 samples (200 seconds)

------------------------------------------------------------
Clique 01 - Run 1/1
------------------------------------------------------------
  Loading model...
  Training data has 13 neurons for this clique
  Filtered training neuron_inf: 13 neurons (from 33 total)
Stage 1: Calibration
Loading first 200 seconds of data...

### 2. Threshold detection (Mountainsort4)
  阈值: 3.5, 间隔: 60, 检测方向: -1, 边界margin: 0
共 16 个通道
Number of detected spikes: 39817

---spike detection rate: 0.8028

Number of matched spikes: 4132
Number of unmatched spikes: 35685

### 3. Extract waveforms (using original traces)
Number of valid spikes: 39810
  GT spikes after time filtering: 5147
  Clique neuron names (eval set, full clique): ['Neuron_101', 'Neuron_226', 'Neuron_59', 'Neuron_64', 'Neuron_65', 'Neuron_66', 'Neuron_95']
  GT spike columns: ['

Noise classification: 100%|██████████| 10/10 [00:00<00:00, 53.51it/s]


Number of spikes passing noise classifier: 5371

---Noise classifier analysis on matched spikes:
  Total matched spikes (from waveform extraction): 591
  Matched spikes passing noise classifier: 507
  Matched spikes rejected by noise classifier: 84
  Matched spike retention rate: 0.8579
  GT spikes after time filtering: 5147
  Clique neuron names (eval set, full clique): ['Neuron_101', 'Neuron_226', 'Neuron_59', 'Neuron_64', 'Neuron_65', 'Neuron_66', 'Neuron_95']
  GT spike columns: ['time', 'neuron']
  GT spikes after neuron filtering (eval neurons, no train filtering): 5147

---spike detection rate (after noise classifier): 0.3732
Number of matched spikes: 1921
Number of unmatched spikes: 3450

### 5. K-means clustering (using way4 features, 30 dimensions, no PCA)
Number of clusters: 18 (Training neurons: 13, additional: 5)
Feature shape: (5371, 30) (way4 features, 30 dimensions)

### 7. Calculate cluster position and waveform (based on train neuron channel_id) and match

Matching re

Noise classification: 100%|██████████| 40/40 [00:00<00:00, 45.99it/s]


Number of spikes passing noise classifier: 41748

---Noise classifier analysis on matched spikes:
  Total matched spikes (from waveform extraction): 8695
  Matched spikes passing noise classifier: 5566
  Matched spikes rejected by noise classifier: 3129
  Matched spike retention rate: 0.6401
  GT spikes after time filtering: 40786
  Clique neuron names (eval set, full clique): ['Neuron_167', 'Neuron_172', 'Neuron_174', 'Neuron_177', 'Neuron_178', 'Neuron_179', 'Neuron_181', 'Neuron_184', 'Neuron_185', 'Neuron_186', 'Neuron_191', 'Neuron_195', 'Neuron_197', 'Neuron_200', 'Neuron_203', 'Neuron_205', 'Neuron_207', 'Neuron_214', 'Neuron_217', 'Neuron_220']
  GT spike columns: ['time', 'neuron']
  GT spikes after neuron filtering (eval neurons, no train filtering): 40786

---spike detection rate (after noise classifier): 0.6054
Number of matched spikes: 24691
Number of unmatched spikes: 17057

### 5. K-means clustering (using way4 features, 30 dimensions, no PCA)
Number of clusters: 23 (Tra

In [6]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [7]:
with open("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique_128/output/clique_01/model_save/run_1/calibration_results.pkl", 'rb') as f:
    a = pickle.load(f)

In [8]:
results_df_simple = a['results_df_simple']
results_df_simple['gt_neuron_label_noise'] = results_df_simple['gt_neuron_label']
results_df_simple['predicted_neuron_label_noise'] = results_df_simple['predicted_neuron_label']
results_df_simple.loc[results_df_simple['gt_neuron_label'] == 'unmatch', 'gt_neuron_label_noise'] = 'noise'
results_df_simple.loc[results_df_simple['predicted_neuron_label'] == 'unmatch', 'predicted_neuron_label_noise'] = 'noise'
